# Challenge 1: Feature Engineering for Fraud Detection

Load raw fraud data from GCS into BigQuery and engineer ML training features
entirely in SQL: one-hot encoding, age binning, an income-to-amount ratio,
days since previous assistance, and boolean-to-integer conversion.

In [ ]:
from google.cloud import bigquery

PROJECT_ID = "qwiklabs-gcp-00-f469c11ef7ce"
DATASET = "fraud_detection"
SOURCE_URI = "gs://labs.roitraining.com/data-to-ai-workshop/fraud_data_raw.csv"

client = bigquery.Client(project=PROJECT_ID, location="US")

In [22]:
RAW_TABLE = f"{PROJECT_ID}.{DATASET}.fraud_data_raw"

client.query(f"CREATE SCHEMA IF NOT EXISTS `{PROJECT_ID}.{DATASET}` OPTIONS(location='US')").result()

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
client.load_table_from_uri(SOURCE_URI, RAW_TABLE, job_config=job_config).result()

table = client.get_table(RAW_TABLE)
print(f"{table.num_rows:,} rows, {len(table.schema)} columns")

50,000 rows, 14 columns


In [23]:
emp_values = [r.Employment_Status for r in client.query(
    f"SELECT DISTINCT Employment_Status FROM `{RAW_TABLE}` WHERE Employment_Status IS NOT NULL"
).result()]

dev_values = [r.Device_Type for r in client.query(
    f"SELECT DISTINCT Device_Type FROM `{RAW_TABLE}` WHERE Device_Type IS NOT NULL"
).result()]

bool_cols = [f.name for f in table.schema if f.field_type == "BOOLEAN"]

print("Employment_Status:", emp_values)
print("Device_Type:", dev_values)
print("Bool columns:", bool_cols)

Employment_Status: ['Unemployed', 'Self-Employed', 'Employed']
Device_Type: ['Mobile', 'Tablet', 'Desktop']
Bool columns: ['Previous_Assistance_Received', 'Supporting_Doc_Verified']


In [24]:
import re

def safe(v):
    return re.sub(r"[^0-9A-Za-z]+", "_", str(v)).strip("_") or "NULL"

emp_cols = [f"IF(Employment_Status = '{v}', 1, 0) AS Employment_Status_{safe(v)}" for v in emp_values]
dev_cols = [f"IF(Device_Type = '{v}', 1, 0) AS Device_Type_{safe(v)}" for v in dev_values]

age_cols = [
    "IF(Age BETWEEN 18 AND 24, 1, 0) AS Age_Bin_18_24",
    "IF(Age BETWEEN 25 AND 34, 1, 0) AS Age_Bin_25_34",
    "IF(Age BETWEEN 35 AND 44, 1, 0) AS Age_Bin_35_44",
    "IF(Age BETWEEN 45 AND 54, 1, 0) AS Age_Bin_45_54",
    "IF(Age BETWEEN 55 AND 64, 1, 0) AS Age_Bin_55_64",
    "IF(Age >= 65, 1, 0) AS Age_Bin_65_plus",
]

bool_int_cols = [f"CAST({c} AS INT64) AS {c}" for c in bool_cols]

transformed = {"Employment_Status", "Device_Type", "Age", *bool_cols}
passthrough = [f.name for f in table.schema if f.name not in transformed]

select_parts = (
    passthrough
    + emp_cols
    + dev_cols
    + age_cols
    + [
        "SAFE_DIVIDE(Income, Amount_Requested) AS Income_to_Amount_Requested",
        "DATE_DIFF(Application_Date, Previous_Assistance_Date, DAY) AS Time_Since_Previous_Assistance",
    ]
    + bool_int_cols
)

OUT_TABLE = f"{PROJECT_ID}.{DATASET}.fraud_training_data"
sql = f"CREATE OR REPLACE TABLE `{OUT_TABLE}` AS\nSELECT\n  " + ",\n  ".join(select_parts) + f"\nFROM `{RAW_TABLE}`"

client.query(sql).result()
print(f"Saved -> {OUT_TABLE}")

Saved -> qwiklabs-gcp-00-f469c11ef7ce.fraud_detection.fraud_training_data


In [25]:
client.query(f"SELECT * FROM `{OUT_TABLE}` LIMIT 5").to_dataframe()


,Applicant_ID,Income,Number_of_Dependents,Amount_Requested,Previous_Assistance_Date,Application_Frequency_Last_Year,IP_Address,Application_Date,Fraudulent,Employment_Status_Unemployed,...,Age_Bin_18_24,Age_Bin_25_34,Age_Bin_35_44,Age_Bin_45_54,Age_Bin_55_64,Age_Bin_65_plus,Income_to_Amount_Requested,Time_Since_Previous_Assistance,Previous_Assistance_Received,Supporting_Doc_Verified
0,1035,60584,5,8349,NaT,1,131.190.190.199,2024-08-22,0,1,...,1,0,0,0,0,0,7.256438,<NA>,0,0
1,1750,0,4,7198,NaT,1,134.91.210.32,2024-03-20,0,1,...,1,0,0,0,0,0,0.000000,<NA>,0,1
2,1803,0,2,3769,NaT,1,228.57.151.146,2024-06-27,0,1,...,0,1,0,0,0,0,0.000000,<NA>,0,0
3,2101,73746,5,2831,NaT,1,147.25.135.145,2024-03-14,0,1,...,0,1,0,0,0,0,26.049452,<NA>,0,1
4,2630,0,2,9521,NaT,1,42.196.209.34,2024-11-08,0,0,...,0,1,0,0,0,0,0.000000,<NA>,0,1


In [26]:
client.query(f"""
SELECT Applicant_ID, Previous_Assistance_Received, Previous_Assistance_Date,
       Application_Date, Time_Since_Previous_Assistance
FROM `{OUT_TABLE}`
WHERE Time_Since_Previous_Assistance IS NOT NULL
LIMIT 5
""").to_dataframe()

,Applicant_ID,Previous_Assistance_Received,Previous_Assistance_Date,Application_Date,Time_Since_Previous_Assistance
0,42837,1,2022-09-13,2024-01-08,482
1,49714,1,2022-11-09,2024-03-10,487
2,45282,1,2022-12-16,2024-01-23,403
3,38133,1,2022-12-19,2024-03-11,448
4,1469,1,2023-02-04,2024-05-22,473
